# Steerling: Logit Contribution Analysis

Decomposes output logits into **known**, **unknown**, and **epsilon** contributions:

$$\text{logit}(y) = \underbrace{\text{known} \cdot W_y}_{\text{known concepts}} + \underbrace{\hat{u} \cdot W_y}_{\text{predicted unknown}} + \underbrace{\varepsilon \cdot W_y}_{\text{residual}}$$

**Requirements:** Interpretable Steerling model, GPU with >= 18 GB VRAM

In [ ]:
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer
from steerling.inference.causal_diffusion import SteerlingGenerator

model_id = "guidelabs/steerling-8b/"


model = AutoModel.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

print(generator)
print(f"Interpretable: {generator.is_interpretable}")

In [ ]:
@torch.no_grad()
def logit_contribution(generator, prompt, gen_length=128, steps=128, temperature=0.0):
    """
    Generate text and decompose each predicted token's logit into
    known, unknown (unk_hat), and epsilon contributions.
    """
    # Generate
    output = generator.generate(prompt, gen_length=gen_length, steps=steps, temperature=temperature)
    prompt_ids = tokenizer.encode(prompt)
    prompt_len = len(prompt_ids)
    text = generator.decode(output, prompt_len=prompt_len)

    # Forward pass with full outputs on the generated sequence
    logits, outputs = generator.model(output,  minimal_output=False)

    # Extract components for the generated region
    gen_slice = slice(prompt_len, output.shape[1])
    known = outputs.known_predicted[0, gen_slice]    # (T_gen, D)
    unk_hat = outputs.unk_hat[0, gen_slice] if outputs.unk_hat is not None else torch.zeros_like(known)
    unk_for_lm = outputs.unk_for_lm[0, gen_slice]   # (T_gen, D)
    epsilon = unk_for_lm - unk_hat                   # (T_gen, D)

    # LM head weight
    lm_weight = generator.model.transformer.lm_head.weight  # (V, D)

    # Predicted token IDs in generated region
    pred_ids = output[0, gen_slice]  # (T_gen,)
    mask_id = generator.mask_token_id
    valid = pred_ids != mask_id

    # W_y for each predicted token: (T_gen, D)
    W_y = lm_weight[pred_ids].float()

    # Dot products: contribution of each component to the predicted token's logit
    known_c = (known.float() * W_y).sum(dim=-1)    # (T_gen,)
    unk_c = (unk_hat.float() * W_y).sum(dim=-1)
    eps_c = (epsilon.float() * W_y).sum(dim=-1)

    # Absolute fractions
    abs_sum = known_c.abs() + unk_c.abs() + eps_c.abs()
    abs_sum = abs_sum.clamp(min=1e-8)

    return {
        "text": text,
        "prompt": prompt,
        "token_ids": pred_ids[valid].cpu(),
        "tokens": [tokenizer.decode([t]) for t in pred_ids[valid].tolist()],
        "known_frac": (known_c.abs() / abs_sum)[valid].cpu().numpy(),
        "unk_frac": (unk_c.abs() / abs_sum)[valid].cpu().numpy(),
        "eps_frac": (eps_c.abs() / abs_sum)[valid].cpu().numpy(),
        "known_c": known_c[valid].cpu().numpy(),
        "unk_c": unk_c[valid].cpu().numpy(),
        "eps_c": eps_c[valid].cpu().numpy(),
    }

In [ ]:
def show_contribution(result, max_tokens=40):
    """Print a compact summary of logit contributions."""
    print(f"Prompt:    {result['prompt']}")
    print(f"Generated: {result['text'][:200]}")
    print()

    n = min(len(result['tokens']), max_tokens)
    kf = result['known_frac'][:n]
    uf = result['unk_frac'][:n]
    ef = result['eps_frac'][:n]

    print(f"{'Token':<15} {'Known':>7} {'Unk':>7} {'Eps':>7}")
    print("-" * 40)
    for i in range(n):
        tok = repr(result['tokens'][i])
        print(f"{tok:<15} {kf[i]:6.1%} {uf[i]:6.1%} {ef[i]:6.1%}")

    print("-" * 40)
    print(f"{'Mean':<15} {kf.mean():6.1%} {uf.mean():6.1%} {ef.mean():6.1%}")

In [ ]:
torch.manual_seed(42)

result = logit_contribution(generator, "The key to understanding artificial intelligence is")
show_contribution(result)